# LegalLens on Colab GPU — improved pipeline (A–D)

Runs the LegalLens summarisation pipeline with the four outcome-preservation fixes:

- **A.** Disposition detection now recognises civil money decrees (`held liable`, `ordered to refund`, `the suit is decreed`, `compensation`/`damages`), not only appeal/criminal forms.
- **B.** The formatter reserves the **judgement** section's words first, so the final order is never trimmed out when a document is over budget.
- **C.** Any operative-order paragraph in the document tail is promoted to a dedicated `judgement` slot — even when section detection falls back to a single block.
- **D.** Evaluation reports two new signals per document: `outcome_present` (did the summary state the order at all?) and `outcome_recall` (token recall of the reference's disposition).

> **Runtime:** set **Runtime → Change runtime type → GPU (T4)** before running.

> **Note on the model:** the fine-tuned checkpoint is loaded from the Hugging Face Hub (`fakechamp24/legallens-bart-base`) by default, so a fresh `git clone` needs no local checkpoint and no training run. The git-ignored local `models/legallens-bart-base-e*` dirs are used only if you set `FINETUNED` to a local path. The inference-time fixes (generation config and Citizen-mode simplification) apply regardless of which model is loaded.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Get the code

**Option A — clone (if you have a git remote).** Fill in your repo URL and branch.

**Option B — upload a zip.** Zip the `legal-doc-intelligence` folder locally, use the Colab file browser (📁 on the left) to upload it, then run:
```python
!unzip -q legal-doc-intelligence.zip
```

**Option C — Google Drive.** `from google.colab import drive; drive.mount('/content/drive')` then `%cd` into the project copy on your Drive.

In [ ]:
import os

# --- Option A: clone ---
REPO_URL = "https://github.com/Phyboc/Legal-Document-Summarizer.git"  # set to "" to use Option B/C
BRANCH   = "review2"   # must be the branch that has the A–D changes + this notebook (commit & push first!)
# Private repo? Clone with a token instead:
#   REPO_URL = "https://<GITHUB_TOKEN>@github.com/Phyboc/Legal-Document-Summarizer.git"

if REPO_URL:
    !rm -rf legallens_repo
    !git clone --branch {BRANCH} {REPO_URL} legallens_repo
    # legal-doc-intelligence is a subfolder of the repo; the pipeline expects it as cwd.
    PROJECT = os.path.abspath("legallens_repo/legal-doc-intelligence")
else:
    # Option B/C: point PROJECT at the uploaded / mounted project root.
    PROJECT = os.path.abspath("legal-doc-intelligence")

# PROJECT is now an ABSOLUTE path. Every later cell runs its shell commands with
# `cd "$PROJECT"` so they work no matter the notebook's current directory (a
# runtime restart or running cells out of order resets it).
assert os.path.isdir(os.path.join(PROJECT, "src", "eval")), (
    f"Project not found at {PROJECT}. The clone failed (check REPO_URL/BRANCH, or the repo "
    "is private and needs a token), or on Option B/C the project isn't uploaded/mounted yet."
)
%cd {PROJECT}
print("PROJECT =", PROJECT)
!ls

## 2. Install dependencies

Colab already ships a CUDA build of PyTorch; this installs the rest from `requirements.txt` plus the spaCy and NLTK data the pipeline needs.

In [ ]:
!cd "{PROJECT}" && pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
print("deps installed")

## 3. (Optional) Build the training dataset

Only needed if you want to **train** on Colab and don't already have `artifacts/training_data/full`. A quick 100-doc build is enough to smoke-test the training path; `--full` is the real build (slow). Skip this cell if you only want to run evaluation.

In [ ]:
# Quick build (100 docs/split). For the real run use: --full --output_dir artifacts/training_data/full
!cd "{PROJECT}" && python scripts/build_dataset.py --sample 100 --output_dir artifacts/training_data/sample100 --overwrite

## 4. (Optional) Fine-tune BART on the GPU

Produces `models/legallens-bart-base-e3`. Point `--data` at the build you made above (`artifacts/training_data/full` for the real model, `artifacts/training_data/sample100` for a smoke test). Skip if you uploaded a checkpoint.

In [ ]:
!cd "{PROJECT}" && python -m src.model.train \
    --data artifacts/training_data/full \
    --output_dir models/legallens-bart-base-e3 \
    --epochs 3

## 5. Run the improved evaluation (A–D)

Scores the full pipeline against the published IN-ABS reference summaries with ROUGE, BERTScore, and the new **outcome** metric. By default the fine-tuned model is pulled from the Hugging Face Hub (`fakechamp24/legallens-bart-base`) and compared against the `facebook/bart-base` zero-shot baseline; set `FINETUNED` to a local path to use a local checkpoint instead.

In [ ]:
import os

# Fine-tuned LegalLens summariser. Default: pull from the Hugging Face Hub, so a
# fresh clone needs no local checkpoint (they are git-ignored) and no training
# run. To use a local checkpoint instead, set FINETUNED to its path, e.g.
#   FINETUNED = os.path.join(PROJECT, "models/legallens-bart-base-e3")
FINETUNED = "fakechamp24/legallens-bart-base"
SAMPLE = 100            # first N test documents; set to None for the whole split
OUTPUT = os.path.join(PROJECT, "artifacts/eval/colab_test.json")   # absolute, so cell 6 always finds it


def _finetuned_available(model: str) -> bool:
    # A local checkpoint directory must actually contain weights; anything else
    # is assumed to be a Hub id that evaluate.py will download.
    if os.path.isdir(model):
        return os.path.exists(os.path.join(model, "model.safetensors"))
    return True


models = ["facebook/bart-base"]
if _finetuned_available(FINETUNED):
    models.append(FINETUNED)
    print("Comparing zero-shot vs fine-tuned:", models)
else:
    print(f"Fine-tuned model {FINETUNED!r} not found - running zero-shot baseline only.")

sample_arg = f"--sample {SAMPLE}" if SAMPLE else ""
!cd "{PROJECT}" && python -m src.eval.evaluate --models {" ".join(models)} --split test {sample_arg} --bertscore --output "{OUTPUT}"

## 6. Show the results (including outcome preservation)

In [ ]:
import json

report = json.load(open(OUTPUT, encoding="utf-8"))
print(f"Split: {report['split']}  |  documents: {report['documents']}\n")

metrics = ["rouge1", "rouge2", "rougeL", "bertscore_f1", "outcome_present", "outcome_recall"]
header = f"{'model':<34}{'mode':<8}" + "".join(f"{m:>16}" for m in metrics)
print(header)
print("-" * len(header))
for r in report["results"]:
    for mode, s in r["scores"].items():
        row = f"{r['model']:<34}{mode:<8}"
        for m in metrics:
            v = s.get(m)
            row += f"{(v if v is not None else '-'):>16}"
        print(row)

print("\noutcome_present = % of summaries that state the court's order;")
print("outcome_recall  = token recall of the reference's disposition. Both are the")
print("metrics that catch a dropped final order — ROUGE/BERTScore barely move on it.")

## 7. (Optional) Save results / checkpoint back to Drive

```python
from google.colab import drive; drive.mount('/content/drive')
!cp artifacts/eval/colab_test.json /content/drive/MyDrive/
# and, to keep a trained checkpoint:
!cp -r models/legallens-bart-base-e3 /content/drive/MyDrive/
```

## 8. Live public demo (free share link)

Launches the Gradio UI and prints a **public `https://*.gradio.live` URL** you can share.
Only needs **steps 1 (clone) and 2 (install)** above — not training or eval. The model is
pulled from the Hub (`fakechamp24/legallens-bart-base`), so no checkpoint is needed.

The link stays live **while this cell runs** (a few hours, until the Colab session ends).
Use a **GPU runtime** (Runtime -> Change runtime type -> GPU) for ~10s summaries; CPU works but is ~60-90s.
Stop the cell to take the demo down.

In [ ]:
# Live public demo. Requires steps 1-2 (PROJECT is set in step 1).
!pip install -q gradio
import os, sys
os.chdir(PROJECT)
sys.path.insert(0, '.')
# Make sure the fine-tuned model is used (Hub repo when no local checkpoint):
os.environ.setdefault('LEGAL_BART_MODEL', 'fakechamp24/legallens-bart-base')
from gradio_app import demo
demo.queue(max_size=8).launch(share=True)